In [2]:
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import torch
import torchaudio

In [3]:
SAMPLE_WAV_FILE = "./sample_audio.wav"

In [4]:
audio, sample_rate = librosa.load(SAMPLE_WAV_FILE)
clip = AudioSegment.from_wav(SAMPLE_WAV_FILE)

bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()
labels = bundle.get_labels()

In [ ]:
class GreedyCTCDecoder(torch.nn.Module):
    def __init__(self, labels):
        super().__init__()
        self.labels = labels
        self.blank = 0

    def forward(self, emission: torch.Tensor) -> str:
        # Pilih label dengan skor tertinggi pada setiap langkah waktu.
        indices = torch.argmax(emission, dim=-1)

        # Gabungkan label sama yang muncul berurutan.
        indices = torch.unique_consecutive(indices, dim=-1)

        # Hapus blank, lalu ubah indeks menjadi karakter.
        indices = [i for i in indices if i != self.blank]
        return "".join([self.labels[i] for i in indices])


decoder = GreedyCTCDecoder(labels=labels)

In [6]:
def get_filename():
    filename = input("New filename (.wav): ")

    while not filename.endswith(".wav"):
        filename = input("Filename must end with .wav: ")

    return filename

In [7]:
def pitch_shift():
    while True:
        try:
            value = float(input("Pitch shift [-5 to 5]: "))
            if -5 <= value <= 5:
                break
        except ValueError:
            pass

        print("Invalid value.")

    filename = get_filename()
    result = librosa.effects.pitch_shift(
        audio, sr=sample_rate, n_steps=value
    )
    sf.write(filename, result, sample_rate)
    print("Saved:", filename)

In [8]:
def white_noise(frames, factor):
    noise = np.random.normal(0, frames.std(), frames.size)
    return frames + noise * factor


def add_white_noise():
    while True:
        try:
            value = float(input("Noise level [0.1 to 0.5]: "))
            if 0.1 <= value <= 0.5:
                break
        except ValueError:
            pass

        print("Invalid value.")

    filename = get_filename()
    result = white_noise(audio, value)
    sf.write(filename, result, sample_rate)
    print("Saved:", filename)

In [9]:
def repeat_clip():
    while True:
        try:
            value = int(input("Repeat count [greater than 0]: "))
            if value > 0:
                break
        except ValueError:
            pass

        print("Invalid value.")

    filename = get_filename()
    result = clip * value
    result.export(filename, format="wav")
    print("Saved:", filename)

In [10]:
def fade_in():
    while True:
        try:
            value = int(input("Fade-in duration in ms [greater than 0]: "))
            if value > 0:
                break
        except ValueError:
            pass

        print("Invalid value.")

    filename = get_filename()
    result = clip.fade_in(value)
    result.export(filename, format="wav")
    print("Saved:", filename)

In [11]:
def predict_text():
    waveform, sr = torchaudio.load(SAMPLE_WAV_FILE)

    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(
            waveform, sr, bundle.sample_rate
        )

    with torch.inference_mode():
        emission, _ = model(waveform)

    transcript = decoder(emission[0])
    print(transcript.replace("|", " "))

In [12]:
def augmentation_menu():
    while True:
        print()
        print("1. Pitch shift")
        print("2. Add white noise")
        print("3. Repeat clip")
        print("4. Insert fade in")
        print("5. Back to main menu")
        choice = input("Choose: ")

        if choice == "1":
            pitch_shift()
        elif choice == "2":
            add_white_noise()
        elif choice == "3":
            repeat_clip()
        elif choice == "4":
            fade_in()
        elif choice == "5":
            break
        else:
            print("Invalid choice.")

In [13]:
def main_menu():
    while True:
        print()
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")
        choice = input("Choose: ")

        if choice == "1":
            augmentation_menu()
        elif choice == "2":
            predict_text()
        elif choice == "3":
            break
        else:
            print("Invalid choice.")

In [ ]:
main_menu()